# Analysis of the final database

This notebook tries to identify characteristics or similarities between the proteins that do have reactions from both TCDB and Rhea.

In [1]:
import pandas as pd
import plotly.express as px

In [5]:
common = pd.read_csv("../1+2/tcdb_rhea_common_reactions.tsv", sep="\t")
tcids_common = common["TCID"]

In [ ]:
parsed_data = []
for entry in tcids_common:
    parts = entry.split(".")
    if len(parts) >= 3:
        new_entry = ".".join(parts[:3])
        parsed_data.append(new_entry)

df = pd.DataFrame(parsed_data, columns=["Class_Subclass_Family"])
df[["Class", "Subclass", "Family"]] = df["Class_Subclass_Family"].str.split(".", expand=True)


fig = px.sunburst(
    df,
    path=["Class", "Subclass", "Family"],
    title="Protein Composition by Class, Subclass, and Family",
    width=700,
    height=700)

fig

Interesting finding on 1.B, very few transporters there have reactions for both Rhea and TCDB.

Also, an analysis of the complete database used in the pipeline, is of an even more obvious importance...

In [16]:
pipeline_db = pd.read_csv("../1+2/transporters_df.tsv", sep="\t")
no_reactions = pipeline_db[pipeline_db["Reaction"] == "[('no_reaction_identified', 'no_reaction_identified')]"]
reactions = pipeline_db[pipeline_db["Reaction"] != "[('no_reaction_identified', 'no_reaction_identified')]"]

In [18]:
def tcid_sunburst(df):
    tc_data = []

    for tc_id in df["TCID"]:
        parts = tc_id.split(".")
        class_ = parts[0]
        subclass = ".".join(parts[:2])
        family = ".".join(parts[:3])
        tc_data.append({"Class": class_, "Subclass": subclass, "Family": family})
    
    df_out = pd.DataFrame(tc_data)
    fig = px.sunburst(df_out, path=["Class", "Subclass", "Family"], color="Class")
    fig.show()

In [ ]:
tcid_sunburst(reactions)
tcid_sunburst(no_reactions)

In [29]:
either_df = pd.read_csv("../1+2/tcdb_rhea_either_reactions.tsv", sep="\t")
filtered_either_df = either_df[
    (either_df["Rhea:Reaction:CHEBI"].notna()) &
    (either_df["TCDB:Reaction:CHEBI"] == "[nan]")
]

In [36]:
filtered_either_df #Remember that this only contians RHEA DATA!!!

,AID,TCID,AA,Reaction,R:Equation,Rhea:Reaction:CHEBI,TCDB:Reaction:CHEBI
19,A0FKN5,2.A.53.2.9,MEDAQESGECLVQNQKYCVERPIYNQEILQGQLHKRERTPQSLRQK...,[nan],['sulfate(out) + chloride(in) = sulfate(in) + ...,['CHEBI:16189 + CHEBI:17996 = CHEBI:16189 + CH...,[nan]
30,A0PJK1,2.A.21.3.15,MAANSTSDLHTPGTQLSVADIIVITVYFALNVAVGIWSSCRASRNT...,[nan],['D-fructopyranose(out) + Na(+)(out) = D-fruct...,['CHEBI:37714 + CHEBI:29101 = CHEBI:37714 + CH...,[nan]
38,A0R049,3.D.3.5.5,MTSAVGTSGTAITSRVHSLNRPNMVSVGTIVWLSSELMFFAGLFAM...,[nan],['4 Fe(II)-[cytochrome c] + O2 + 8 H(+)(in) = ...,['4 CHEBI:29033 + CHEBI:15379 + 8 CHEBI:15378 ...,[nan]
39,A0R050,3.D.3.5.5,MTSKSRRRLRRRLSAGLLLLIGLAVAGGVAATLTPQPQVAVADESQ...,[nan],['a quinol + 2 Fe(III)-[cytochrome c](out) = a...,['CHEBI:24646 + 2 CHEBI:29034 = CHEBI:132124 +...,[nan]
40,A0R052,3.D.3.5.5,MSPDFAKLAAAQGDAIDSRYHPSAAVRRQLNKVFPTHWSFLLGEIA...,[nan],['a quinol + 2 Fe(III)-[cytochrome c](out) = a...,['CHEBI:24646 + 2 CHEBI:29034 = CHEBI:132124 +...,[nan]
...,...,...,...,...,...,...,...
8616,XP_002452437.1,9.B.115.2.1,MVSMAAAAAAVGVLLPFPFYYALWTHPQRWVDLCGRGADPCRRMAQ...,[nan],"['a 1,2-diacyl-sn-glycero-3-phospho-N,N-dimeth...",['CHEBI:64572 + CHEBI:59789 = CHEBI:57643 + CH...,[nan]
8621,XP_007449084.1,8.A.176.1.3,MATEGMILTNHDHQIRVGVLTVSDSCFRNLAEDRSGINLKDLVQDP...,[nan],['molybdopterin + ATP + H(+) = adenylyl-molybd...,['CHEBI:58698 + CHEBI:30616 + CHEBI:15378 = CH...,[nan]
8626,XP_011409656.2,1.A.114.1.3,MADRSSIHFVPASLPPQLERGRVREDSWKNEMNSFARGRPVSEADA...,[nan],['chloride(in) = chloride(out)'],['CHEBI:17996 = CHEBI:17996'],[nan]
8631,XP_013816093.1,9.A.79.1.2,MLIGEIFELMQFIFVVAFTTFLISCVDYDILFANKAVNHSQHPSEP...,[nan],"['a 1,2-diacyl-sn-glycero-3-phosphocholine(in)...","['CHEBI:57643 = CHEBI:57643', 'CHEBI:57262 = C...",[nan]


In [37]:
all_df = pd.read_csv("../1+2/all.tsv", sep="\t")
merged_df = pd.merge(
    either_df,
    all_df[["AID", "TCID", "AA"]],
    on="AID",
    how="outer",
    suffixes=("_filtered", "_eq")
)

merged_df.drop_duplicates(subset=["AID", "TCID_eq", "AA_eq"], keep="first", inplace=True)
merged_df["TCID_filtered"] = merged_df["TCID_filtered"].fillna(merged_df["TCID_eq"])
merged_df["AA_filtered"] = merged_df["AA_filtered"].fillna(merged_df["AA_eq"])
merged_df.drop(columns=["TCID_eq", "AA_eq"], inplace=True)
merged_df.rename(columns={"TCID_filtered": "TCID","AA_filtered": "AA"}, inplace=True)
merged_df

,AID,TCID,AA,Reaction,R:Equation,Rhea:Reaction:CHEBI,TCDB:Reaction:CHEBI
0,1CN3_F,1.A.83.1.5,GGGGGGGGAASHQRVTPDWMLPLILGLYG,NaN,NaN,NaN,NaN
1,2IMU_A,1.A.59.1.2,FGFKDIIRAIRRIAVPVVSTLFPPAAPLAHAIGEGVDYLLGDEAQA,NaN,NaN,NaN,NaN
2,2MUN_A,8.B.26.1.3,ADNKCENSLRREIACGQCRDKVKTDGYFYECCTSDSTFKKCQDLLH,NaN,NaN,NaN,NaN
3,3IO0_A,1.S.2.1.3,APTMTEFVGTAGGDTVGLVIANVDSLLHKHLGLDNTCRSIGIISAR...,NaN,NaN,NaN,NaN
4,3KBC_A,2.A.23.1.15,MGLYRKYIEYPVLQKILIGLILGAIVGLILGHYGYAHAVHTYVKPF...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
58354,s0EYI0,4.D.1.4.4,MKEAIVLARLLLDAGLLGYLAYACLTVVAAWQWRHSAIPPNQPLAL...,NaN,NaN,NaN,NaN
58355,s6ex81,2.A.3.3.23,MGFMRKADFELYRDADKHYNQVLTTRDFLALGVGTIISTSIFTLPG...,['L-isoleucine (out) + n H+ (out) = L-isoleuci...,[nan],[nan],['CHEBI:17191 + n CHEBI:15378 = CHEBI:17191 + ...
58361,wp_147664752,2.A.7.43.6,MNSTLIAIFEILIGVGLIGFWIYFFLVENKNPEKSKVYLGFERSFP...,NaN,NaN,NaN,NaN
58362,xp_008873677,1.A.17.1.24,MLQGDEESAPLVSSQTRADGNPALLDGVNGDSQVDAPSPTFAEPVV...,['chloride (out) = chloride (in)'],[nan],[nan],['CHEBI:17996 = CHEBI:17996']


In [ ]:








tcid_sunburst(filtered_either_df)